# 2 · Guainía: tomography with the L1c LUTs

**Pipeline:** 0 · Data access → 1 · Stack → **2 · Tomography** → 3 · Forest height → 4 · Polarimetric analysis

Input: the stack and `Guainia_outputs/run_manifest.json` from notebook 1.

Tomography focuses the stack in height. For every multilook cell, the $N$ acquisitions form a vector $\mathbf y$ whose phases depend on the heights of the scatterers through the vertical wavenumbers $k_z^i$. Before that relation holds, two phase terms supplied in the LUT must be removed:

| LUT layer (stack variable) | what it removes | after removal |
|---|---|---|
| flattening phase screen (`dem_phases`) | phase of the DEM surface relative to the primary (topography + flat earth) | phase relative to the DEM |
| SKP calibration phase screen (`calibration_phases`) | residual phase errors common to all heights (orbit, ionosphere, troposphere, DEM error), estimated from the ground term of a Sum-of-Kronecker-Products decomposition | **ground at $z = 0$** |
| vertical wavenumbers (`insar_wavenumbers`) | | steering vector $a_i(z)=e^{-jk_z^i z}$ |

$$s^i_{corr} = s^i\,e^{j(\mathrm{SIGN\_DEM}\,\phi^i_{flat} + \mathrm{SIGN\_CAL}\,\phi^i_{SKP})}$$

**Steps**
1. load the crop; reference = coregistration primary;
2. show the interferometric phase before and after each LUT correction;
3. determine the LUT phase signs from the data;
4. multibaseline coherence matrices per polarimetric channel;
5. $k_z$ sign check, beamformer and Capon tomograms;
6. vertical profiles, transects and horizontal slices;
7. save tomograms and coherences; record the phase conventions in the manifest.

## 1. Configuration
`WINDOW_M` is the multilook window **on the ground** in both directions. Slant-range spacing is converted with the incidence angle (19.8 m slant ≈ 47 m ground at 25°). Coherence and tomograms are computed on a grid decimated by half the window.

In [ ]:
import pathlib, sys
SITE_DIR = pathlib.Path.cwd()                       # the Guainia/ folder (notebooks run from here)
if not (SITE_DIR / "stack_download").is_dir():
    raise FileNotFoundError("open the notebook from the Guainia/ folder (stack_download/ not found)")
ROOT = next(p for p in [SITE_DIR, *SITE_DIR.parents] if (p / "biomass_tomo").is_dir())
sys.path.insert(0, str(ROOT))                       # the biomass_tomo package

import time
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter
import warnings
warnings.filterwarnings("ignore", message="(All-NaN slice|Mean of empty slice|invalid value|divide by zero)", category=RuntimeWarning)

from biomass_tomo import stack as st, tomo

MANIFEST = SITE_DIR / "Guainia_outputs" / "run_manifest.json"
CROP = None                                  # None = crop in the manifest (default: whole stack); or (az0, az1, rg0, rg1)
WINDOW_M = 200.0                             # multilook window on the ground [m]
Z_VECTOR = np.arange(-30, 81, 2.0)           # tomographic height axis above ground [m]
D_LOAD = 0.0                                 # Capon diagonal loading
CHANNELS = {"HH": ["HH"], "VV": ["VV"], "X": ["HV", "VH"]}   # X = (HV + VH)/2, symmetrised cross-pol
SIGN_BLOCK = 3000                            # azimuth lines (crop centre) used to determine the LUT signs

cfg = st.read_manifest(MANIFEST)
CROP = list(CROP or cfg["crop"])
OUT = pathlib.Path(cfg["out_dir"]) / "tomography"
OUT.mkdir(parents=True, exist_ok=True)
print(cfg["site"], cfg["stack_key"], "| crop", CROP)

## 2. Data
The crop is read into the course layout (range, azimuth, acquisition). The full frame needs about 12 GB of memory.

In [ ]:
t0 = time.time()
ds = st.open_stack(cfg["stack_path"])
REF = st.reference_index(ds)
data = st.load_crop(ds, CROP)
slc, aux = data["slc"], data["aux"]
r_ax, x_ax, acq = data["r_ax"], data["x_ax"], data["acquisition"]
dr, dx = r_ax[1] - r_ax[0], x_ax[1] - x_ax[0]
Nr, Nx, Nb = aux["insar_wavenumbers"].shape
THETA = np.deg2rad(np.nanmean(aux["incidence_angle"]))
window_r, window_x = tomo.window_samples(WINDOW_M, dr / np.sin(THETA), dx)
print(f"loaded in {time.time() - t0:.0f} s: {Nr} range x {Nx} azimuth, {Nb} acquisitions; "
      f"reference index {REF} (C{int(acq[REF]):02d})")
print(f"slant spacing {dr:.2f} m (ground {dr / np.sin(THETA):.1f} m), azimuth {dx:.2f} m, mean incidence {np.rad2deg(THETA):.1f} deg")
print(f"multilook window {window_r} x {window_x} samples (range x azimuth) = {window_r * window_x} looks")

## 3. What the LUT corrections do
HH interferometric phase between the primary and the acquisition with the largest baseline, multilooked over a central block, at three stages:
1. raw: topography plus residual phase errors;
2. after the flattening screen: residual phase errors plus the height of the scattering phase centre;
3. after the SKP screen: only $k_z\,z_{pc}$ remains, where $z_{pc}$ is the phase-centre height above the ground. It is nearly uniform and its coherence is highest.

The panel uses the signs found in section 4 (the cell can be re-run after that section). Until then it uses +1.

In [ ]:
kz_scene = np.nanmean(aux["insar_wavenumbers"][::20, ::200], axis=(0, 1))
FAR = int(np.argmax(np.abs(kz_scene)))
c0 = Nx // 2
blk = slice(max(c0 - SIGN_BLOCK // 2, 0), min(c0 + SIGN_BLOCK // 2, Nx))
sd_, sc_ = globals().get("SIGN_DEM", 1), globals().get("SIGN_CAL", 1)

def multilooked_ifg(a, b):
    p = a * np.conj(b)
    w = (window_r, window_x)
    c = uniform_filter(p.real, w) + 1j * uniform_filter(p.imag, w)
    n = np.sqrt(uniform_filter(np.abs(a) ** 2, w) * uniform_filter(np.abs(b) ** 2, w))
    return c / n

s_ref, s_far = slc["HH"][:, blk, REF], slc["HH"][:, blk, FAR]
ph = {i: (sd_ * aux["dem_phases"][:, blk, i], sc_ * aux["calibration_phases"][:, blk, i]) for i in (REF, FAR)}
stages = {"raw": (s_ref, s_far),
          "+ flattening": (s_ref * np.exp(1j * ph[REF][0]), s_far * np.exp(1j * ph[FAR][0])),
          "+ flattening + SKP": (s_ref * np.exp(1j * (ph[REF][0] + ph[REF][1])), s_far * np.exp(1j * (ph[FAR][0] + ph[FAR][1])))}
ext = [x_ax[blk][0] / 1e3, x_ax[blk][-1] / 1e3, r_ax[0] / 1e3, r_ax[-1] / 1e3]
fig, axes = plt.subplots(len(stages), 2, figsize=(13, 2.3 * len(stages)), sharex=True, constrained_layout=True)
for row, (name, (a, b)) in enumerate(stages.items()):
    g = multilooked_ifg(a, b)
    im0 = axes[row, 0].imshow(np.angle(g), aspect="auto", origin="lower", cmap="twilight", vmin=-np.pi, vmax=np.pi, extent=ext)
    im1 = axes[row, 1].imshow(np.abs(g), aspect="auto", origin="lower", cmap="gray", vmin=0, vmax=1, extent=ext)
    R = np.abs(np.nanmean(np.exp(1j * np.angle(g))))
    axes[row, 0].set_title(f"phase, {name}  (uniformity R = {R:.2f})", fontsize=8, loc="left")
    axes[row, 1].set_title(f"|coherence|, {name}  (mean {np.nanmean(np.abs(g)):.2f})", fontsize=8, loc="left")
    axes[row, 0].set_ylabel("slant range [km]")
fig.colorbar(im0, ax=axes[:, 0], shrink=0.6, label="rad"); fig.colorbar(im1, ax=axes[:, 1], shrink=0.6)
for ax in axes[-1]: ax.set_xlabel("azimuth [km]")
fig.suptitle(f"HH, C{int(acq[REF]):02d} × C{int(acq[FAR]):02d}, mean $k_z$ = {kz_scene[FAR]:.3f} rad/m", fontsize=10)
plt.show()
del s_ref, s_far, stages

## 4. LUT phase signs
The sign convention of the two screens is not documented, so both are found from the data on the temporal-neighbour pair (the shortest baseline, with little volume decorrelation), over the central `SIGN_BLOCK` lines.
- **Flattening sign:** the sign that maximises mean coherence. The wrong sign leaves twice the topographic phase and decorrelates the multilook.
- **SKP sign:** the sign that maximises the interferogram uniformity $R = |\langle e^{j\arg\gamma}\rangle|$, since after calibration only $k_z z_{pc}$ remains.

In [ ]:
SEC = REF - 1 if REF > 0 else REF + 1
SIGN_DEM, SIGN_CAL, scores, margins = tomo.phase_signs(slc["HH"][:, blk], aux["dem_phases"][:, blk], aux["calibration_phases"][:, blk],
                                                       REF, SEC, window_r, window_x)
for (sd, sc), s in sorted(scores.items(), key=lambda kv: -kv[1]["coherence"]):
    print(f"SIGN_DEM {sd:+d} SIGN_CAL {sc:+d}: mean|coh| {s['coherence']:.3f}  uniformity {s['uniformity']:.3f}")
print(f"adopted SIGN_DEM {SIGN_DEM:+d}, SIGN_CAL {SIGN_CAL:+d}; margins {margins}")
if min(margins.values()) < 0.05:
    print("WARNING: small margin, the sign is poorly determined on this block")

In [ ]:
corr = tomo.phase_correction(aux["dem_phases"], aux["calibration_phases"], SIGN_DEM, SIGN_CAL)
for p in slc:
    slc[p] *= corr
del corr
print("SLCs phase-corrected (flattening + SKP)")

## 5. Multibaseline coherence
For each channel, the $N\times N$ coherence matrix
$$\hat\Gamma_{mn} = \frac{\langle y_m y_n^*\rangle}{\sqrt{\langle|y_m|^2\rangle\langle|y_n|^2\rangle}}$$
is estimated over the `WINDOW_M` boxcar and decimated by half the window. Cells with less than 90 % valid samples are NaN. The result is cached, so re-running is fast.

In [ ]:
tag = f"az{CROP[0]}-{CROP[1]}_rg{CROP[2]}-{CROP[3]}_w{int(WINDOW_M)}_s{SIGN_DEM:+d}{SIGN_CAL:+d}"
coherence, intensity, coh_files = {}, {}, {}
for ch, pols in CHANNELS.items():
    cache = OUT / f"coherence_{ch}_{tag}.npz"
    coh_files[ch] = str(cache)
    if cache.exists():
        z = np.load(cache)
        coherence[ch], intensity[ch], ds_r, ds_x = z["coherence"], z["intensity"], int(z["ds_r"]), int(z["ds_x"])
    else:
        t0 = time.time()
        channel = slc[pols[0]] if len(pols) == 1 else sum(slc[p] for p in pols) / len(pols)
        coherence[ch], intensity[ch], ds_r, ds_x = tomo.coherence_matrix(channel, window_r, window_x)
        del channel
        np.savez(cache, coherence=coherence[ch], intensity=intensity[ch], ds_r=ds_r, ds_x=ds_x)
        print(f"{ch}: computed in {time.time() - t0:.0f} s")
    off = ~np.eye(Nb, dtype=bool)
    print(f"{ch}: coherence grid {coherence[ch].shape[:2]}, mean |gamma| off-diagonal {np.nanmean(np.abs(coherence[ch])[:, :, off]):.2f}")
del slc
Nr_c, Nx_c = coherence["X"].shape[:2]
sub_ds = lambda a: a[::ds_r, ::ds_x][:Nr_c, :Nx_c]
r_ds, x_ds = r_ax[::ds_r][:Nr_c], x_ax[::ds_x][:Nx_c]
kz_raw = sub_ds(aux["insar_wavenumbers"])
lat_ds, lon_ds, inc_ds, dem_ds = (sub_ds(aux[k]) for k in ("latitude", "longitude", "incidence_angle", "dem"))

Mean |coherence| matrices: coherence falls with the $k_z$ separation (volume decorrelation) and, less, with time. The right panel plots every pair against $|\Delta k_z|$: the decay rate reflects the vertical extent of the scatterers in each channel.

In [ ]:
fig, axes = plt.subplots(1, len(CHANNELS) + 1, figsize=(4 * (len(CHANNELS) + 1), 3.6), constrained_layout=True)
labels = [f"C{int(a):02d}" for a in acq]
for ax, ch in zip(axes, CHANNELS):
    im = ax.imshow(np.nanmean(np.abs(coherence[ch]), axis=(0, 1)), vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(Nb), labels, rotation=90, fontsize=7); ax.set_yticks(range(Nb), labels, fontsize=7); ax.set_title(f"{ch}: mean |γ|", fontsize=9)
fig.colorbar(im, ax=axes[:-1], shrink=0.8)
kzm = np.nanmean(kz_raw, axis=(0, 1))
for ch in CHANNELS:
    m = np.nanmean(np.abs(coherence[ch]), axis=(0, 1))
    pairs = [(i, j) for i in range(Nb) for j in range(i + 1, Nb)]
    axes[-1].plot([abs(kzm[i] - kzm[j]) for i, j in pairs], [m[i, j] for i, j in pairs], "o", label=ch, alpha=0.7)
axes[-1].set_xlabel(r"$|\Delta k_z|$ [rad/m]"); axes[-1].set_ylabel("mean |γ|"); axes[-1].legend(); axes[-1].grid(alpha=0.3)
plt.show()

## 6. Tomograms and the $k_z$ sign
- Beamformer: $P_{BF}(z) = \mathbf a^H \hat{\mathbf\Gamma}\,\mathbf a$
- Capon: $P_{CP}(z) = 1/(\mathbf a^H\hat{\mathbf\Gamma}^{-1}\mathbf a)$

Here $\{\mathbf a\}_i = e^{-jk_z^i z}$. Using normalised coherence (not covariance) equalises the acquisitions. After the SKP calibration the ground is at $z = 0$ and the forest lies above it. If the mean profiles put more power below the ground than above, the $k_z$ sign is flipped.

In [ ]:
t0 = time.time()
KZ_SIGN = +1
for attempt in range(2):
    kz = KZ_SIGN * kz_raw
    tomos = {ch: tomo.tomograms(coherence[ch], kz, Z_VECTOR, D_LOAD) for ch in CHANNELS}
    profiles = {ch: np.nanmean(tomos[ch]["Beamformer"], axis=(0, 1)) for ch in CHANNELS}
    ratios, ok = tomo.kz_sign_check(profiles, Z_VECTOR)
    print(f"KZ_SIGN {KZ_SIGN:+d}: beamformer power above / below ground {ratios}")
    if ok:
        break
    KZ_SIGN = -KZ_SIGN
print(f"adopted KZ_SIGN {KZ_SIGN:+d} ({time.time() - t0:.0f} s)")
summary = tomo.kz_summary(kz)
print(f"vertical resolution {summary['resolution_m']:.1f} m, height of ambiguity {summary['ambiguity_m']:.1f} m")

### Mean vertical profiles
Median and interquartile range over all valid cells, each profile normalised to its maximum.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True, constrained_layout=True)
for ax, method in zip(axes, ("Beamformer", "Capon")):
    for ch in CHANNELS:
        t = tomos[ch][method].reshape(-1, Z_VECTOR.size)
        t = t / np.nanmax(t, axis=1, keepdims=True)
        q = np.nanpercentile(t, [25, 50, 75], axis=0)
        line, = ax.plot(q[1], Z_VECTOR, label=ch)
        ax.fill_betweenx(Z_VECTOR, q[0], q[2], color=line.get_color(), alpha=0.15)
    ax.axhline(0, color="k", lw=0.8); ax.set_title(f"{method} profile (normalised)"); ax.set_xlabel("normalised power"); ax.grid(alpha=0.3)
axes[0].set_ylabel("height above ground [m]"); axes[1].legend()
plt.show()

### Vertical transects
Capon tomograms along azimuth at mid range (top three panels) and along range at mid azimuth (bottom panel). Each column is normalised to its maximum. The white dashed line is the SKP ground ($z = 0$).

In [ ]:
R_SLICE, X_SLICE = Nr_c // 2, Nx_c // 2
dz = Z_VECTOR[1] - Z_VECTOR[0]
fig, axes = plt.subplots(len(CHANNELS) + 1, 1, figsize=(12, 2.3 * (len(CHANNELS) + 1)), constrained_layout=True)
for ax, ch in zip(axes, CHANNELS):
    t = tomos[ch]["Capon"][R_SLICE].T
    im = ax.imshow(t / np.nanmax(t, axis=0, keepdims=True), aspect="auto", origin="lower", cmap="jet", vmin=0, vmax=1,
                   extent=[x_ds[0] / 1e3, x_ds[-1] / 1e3, Z_VECTOR[0] - dz / 2, Z_VECTOR[-1] + dz / 2], interpolation="none")
    ax.axhline(0, color="w", ls="--", lw=0.8); ax.set_ylabel("height [m]")
    ax.set_title(f"Capon {ch}, azimuth transect at slant range {r_ds[R_SLICE] / 1e3:.1f} km", fontsize=8, loc="left")
axes[len(CHANNELS) - 1].set_xlabel("azimuth [km]")
t = tomos["X"]["Capon"][:, X_SLICE].T
axes[-1].imshow(t / np.nanmax(t, axis=0, keepdims=True), aspect="auto", origin="lower", cmap="jet", vmin=0, vmax=1,
                extent=[r_ds[0] / 1e3, r_ds[-1] / 1e3, Z_VECTOR[0] - dz / 2, Z_VECTOR[-1] + dz / 2], interpolation="none")
axes[-1].axhline(0, color="w", ls="--", lw=0.8); axes[-1].set_ylabel("height [m]"); axes[-1].set_xlabel("slant range [km]")
axes[-1].set_title(f"Capon X, range transect at azimuth {x_ds[X_SLICE] / 1e3:.1f} km", fontsize=8, loc="left")
fig.colorbar(im, ax=axes, shrink=0.5, label="column-normalised power")
plt.show()

### Horizontal slices
Fraction of the cross-pol Capon power at selected heights, each pixel normalised by its total power over 0–60 m. Ground-dominated areas (bare soil, water, savanna) are bright at 0 m, tall forest at 20–30 m.

In [ ]:
SLICES = [0, 10, 20, 30, 40]
t = tomos["X"]["Capon"]
total = np.nansum(t[..., (Z_VECTOR >= 0) & (Z_VECTOR <= 60)], axis=-1)
fig, axes = plt.subplots(len(SLICES), 1, figsize=(12, 1.6 * len(SLICES)), sharex=True, constrained_layout=True)
for ax, h in zip(axes, SLICES):
    k = int(np.argmin(np.abs(Z_VECTOR - h)))
    im = ax.imshow(t[..., k] / total, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=np.nanpercentile(t[..., k] / total, 98),
                   extent=[x_ds[0] / 1e3, x_ds[-1] / 1e3, r_ds[0] / 1e3, r_ds[-1] / 1e3], interpolation="none")
    ax.set_title(f"z = {Z_VECTOR[k]:.0f} m", fontsize=8, loc="left"); ax.set_ylabel("range [km]")
    fig.colorbar(im, ax=ax, pad=0.01, aspect=8)
axes[-1].set_xlabel("azimuth [km]")
plt.show()

## 7. Saving
- `tomography/tomograms_<tag>.nc`: Capon and beamformer tomograms per channel; multilooked intensity; $k_z$, incidence angle, DEM, latitude and longitude on the coherence grid.
- `tomography/coherence_<ch>_<tag>.npz`: coherence matrices (used by notebook 3 for the phase-centre and RVoG heights).
- The manifest gains `phase` (signs, reference) and `tomography` (file paths, window), so notebooks 3 and 4 use exactly the same conventions.

In [ ]:
grid = ("range", "azimuth")
tomo_ds = xr.Dataset(
    {**{f"capon_{ch}": (grid + ("height",), tomos[ch]["Capon"]) for ch in CHANNELS},
     **{f"beamformer_{ch}": (grid + ("height",), tomos[ch]["Beamformer"]) for ch in CHANNELS},
     **{f"intensity_{ch}": (grid, intensity[ch].astype(np.float32)[:Nr_c, :Nx_c]) for ch in CHANNELS},
     "kz": (grid + ("acquisition",), kz.astype(np.float32)),
     "incidence_angle": (grid, inc_ds), "dem": (grid, dem_ds), "latitude": (grid, lat_ds), "longitude": (grid, lon_ds)},
    coords={"range": r_ds, "azimuth": x_ds, "height": Z_VECTOR, "acquisition": acq},
    attrs={"stack": cfg["stack_path"], "crop": str(CROP), "window_m": WINDOW_M, "window_samples": f"{window_r} x {window_x}",
           "decimation": f"{ds_r} x {ds_x}", "sign_dem": SIGN_DEM, "sign_cal": SIGN_CAL, "kz_sign": KZ_SIGN,
           "reference_index": REF, "height_reference": "ground (after SKP calibration)",
           "kz_definition": "steering a_i(z) = exp(-j kz_i z), kz already multiplied by kz_sign"})
tomo_ds["range"].attrs["units"] = tomo_ds["azimuth"].attrs["units"] = tomo_ds["height"].attrs["units"] = "m"
tomo_path = OUT / f"tomograms_{tag}.nc"
st.save_netcdf(tomo_ds, tomo_path)

cfg.update(crop=CROP,
           phase={"sign_dem": SIGN_DEM, "sign_cal": SIGN_CAL, "kz_sign": KZ_SIGN, "reference_index": REF, "sign_margins": margins},
           tomography={"tomograms": str(tomo_path), "coherence": coh_files, "window_m": WINDOW_M, "tag": tag,
                       "resolution_m": float(summary["resolution_m"]), "ambiguity_m": float(summary["ambiguity_m"])})
st.write_manifest(MANIFEST, **cfg)
print(f"saved {tomo_path} ({tomo_path.stat().st_size / 1e9:.2f} GB) and updated {MANIFEST}")